# Atividade 1 – Segmentação e Visão Computacional


Seções:
- Parte I: Limiarização (manual, Otsu, adaptativa)
- Parte II: Espaços de cor (RGB, HSV, L*a*b*)
- Parte III: Agrupamento k-means no plano a*b*
- Parte IV: Morfologia e refino das máscaras
- Análise comparativa final


In [ ]:
from pathlib import Path

import numpy as np
import cv2
from skimage import color
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (6, 4)

DATA_DIR = Path("images") / "input"
RESULTS_DIR = Path("images") / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


ModuleNotFoundError: No module named 'src'

## Carregamento de imagem

Aqui ficam funções auxiliares para leitura e exibição das imagens da pasta `images/input`.


In [3]:
# carregamento da imagem RGB da pasta images/input
def load_rgb(name: str):
    """Carrega uma imagem RGB da pasta `images/input`."""
    path = DATA_DIR / name
    img_bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img_bgr is None:
        raise FileNotFoundError(f"Imagem não encontrada: {path}")
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    return img_rgb

# show imagens lado a lado
def show_side_by_side(images, titles):
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        axes = [axes]
    for ax, img, title in zip(axes, images, titles):
        if img.ndim == 2:
            ax.imshow(img, cmap="gray")
        else:
            ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()


## Parte I – Limiarização (exemplo com Otsu)

Este bloco demonstra como chamar a função `otsu_threshold` (a ser implementada em `src/otsu.py`).


In [ ]:
img_rgb = load_rgb("tomate.png")  # ajuste o nome do arquivo se necessário
img_gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

t = otsu_threshold(img_gray)
print("Limiar de Otsu:", t)

mask = img_gray >= t

show_side_by_side(
    [img_rgb, img_gray, mask],
    ["Original", "Tons de cinza", "Máscara (Otsu)"]
)


## Parte III – K-means no espaço L*a*b*

Exemplo de uso da função `kmeans_lab_ab` (a ser implementada em `src/kmeans.py`).


In [ ]:
n_clusters = 3  # por exemplo: tomate, folha, fundo

lab = color.rgb2lab(img_rgb)
ab = lab[..., 1:3].reshape(-1, 2)

labels, centers = kmeans_lab_ab(ab, n_clusters=n_clusters)
labels_img = labels.reshape(img_rgb.shape[:2])

plt.figure(figsize=(5, 4))
plt.imshow(labels_img, cmap="tab10")
plt.title("Segmentação por k-means (a*b*)")
plt.axis("off")
plt.tight_layout()


## Parte IV – Morfologia

Exemplo de aplicação de abertura e fechamento sobre uma máscara binária.


In [ ]:
kernel_size = 5

mask_open = opening(mask, kernel_size)
mask_closed = closing(mask_open, kernel_size)

show_side_by_side(
    [mask, mask_open, mask_closed],
    ["Máscara original", "Após abertura", "Após fechamento"]
)
